# Parte 7: Preprocesamiento
## Riesgo de Contaminación por Nitratos | La Rioja, 2015 - 2025

*Objetivo:* Preparar la base de datos para el modelado, asegurando la consistencia metodológica de las variables y previniendo cualquier fuga de información (data leakage). Para ello, se estructuran primero los metadatos de los registros, se separa el conjunto en entrenamiento (80%) y prueba (20%), y se aplican de forma rigurosa las técnicas de imputación, codificación y escalado ajustándolas únicamente con el conjunto de entrenamiento.

### 1. Importación de Librerías y Configuración General
Cargamos las librerías esenciales para manipulación de datos, división train/test, imputación de nulos y normalización de variables, además de librerías para guardar transformadores y configuraciones.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib
import json

### 2. Carga del Dataset e Identificación Inicial
Cargamos el conjunto de datos integrado que reúne las variables climáticas, edáficas, de cobertura vegetal y SIGPAC para el periodo 2015-2025.

In [2]:
BASE_DIR = Path(r"C:\Users\mcangulo\OneDrive - FEDERACION DE EMPRESAS DE LA RIOJA\Escritorio\dataset_larioja")
ruta_excel = BASE_DIR / "9_dataset_final" / "dataset_final_integrado_larioja_2015_2025.xlsx"

df = pd.read_excel(ruta_excel)
df_clean = df.copy()
print("Shape inicial:", df_clean.shape)

Shape inicial: (1173, 105)


### 3. Extracción de Metadatos y Estructuración
Antes de proceder a la eliminación de columnas identificadoras o realizar la división de datos, creamos una tabla de metadatos asociada a cada observación para conservar variables clave como `ipa` (punto de monitoreo), coordenadas y año/fecha, las cuales se utilizarán en validación temporal, validación espacial o generación de mapas. Creamos además un identificador estable `id_registro` basado en el índice.

In [3]:
# Creación de un identificador estable
df_clean["id_registro"] = df_clean.index

# Renombrado homogéneo de coordenadas de latitud/longitud principales
if "lat_x" in df_clean.columns and "lon_x" in df_clean.columns:
    df_clean = df_clean.rename(columns={"lat_x": "lat", "lon_x": "lon"})

# Selección y copia de variables de metadatos
metadata_cols = ["id_registro", "ipa", "anio", "fecha", "lat", "lon"]
existing_meta_cols = [c for c in metadata_cols if c in df_clean.columns]

metadata = df_clean[existing_meta_cols].copy()
print(f"Metadatos construidos con {metadata.shape[1]} variables y {metadata.shape[0]} observaciones.")

Metadatos construidos con 6 variables y 1173 observaciones.


### 4. Eliminación de Columnas Identificadoras Redundantes
Eliminamos columnas secundarias, de texto o identificadores espaciales de la estación meteorológica, ya que no se utilizarán como variables predictivas en el modelado.

In [4]:
cols_identificadores = [
    'ipa', 'nombre_punto', 'codigo_masa_agua', 'masa_agua',
    'indicativo', 'nombre', 'indicativo_clima', 'nombre_clima',
    'nombre_punto_esp', 'masa_agua_esp', 'sigpac_uso',
    'municipio_esp', 'provincia_esp', 'red_esp', 'provincia',
    'anio_clima', 'mes_clima', 'clc2012_codigo', 'clc2018_codigo',
    'latitud', 'longitud', 'x_utm30_esp', 'y_utm30_esp', 'lat_y', 'lon_y'
]
cols_identificadores = [c for c in cols_identificadores if c in df_clean.columns]
df_clean = df_clean.drop(columns=cols_identificadores)
print(f"Eliminadas {len(cols_identificadores)} columnas identificadoras. Shape actual: {df_clean.shape}")

Eliminadas 25 columnas identificadoras. Shape actual: (1173, 81)


### 5. Eliminar Columnas con Excesiva Tasa de Nulos
Las variables `localidad` y `cota` presentan una tasa de valores faltantes extremadamente alta, por lo que se descartan para evitar imputaciones poco fiables.

In [5]:
cols_incompletas = ['localidad', 'cota']
cols_incompletas = [c for c in cols_incompletas if c in df_clean.columns]
df_clean = df_clean.drop(columns=cols_incompletas)
print(f"Eliminadas variables con excesiva tasa de nulos: {cols_incompletas}. Shape: {df_clean.shape}")

Eliminadas variables con excesiva tasa de nulos: ['localidad', 'cota']. Shape: (1173, 79)


### 6. Depurar Filas con Nulos Climáticos Residuales
Se detectó un único registro con valores climáticos nulos residuales. Dado el bajo impacto en la muestra, se elimina dicha observación para no distorsionar el conjunto.

In [6]:
cols_clima_residual = ['precip_30d', 'temp_media_30d']
cols_clima_residual = [c for c in cols_clima_residual if c in df_clean.columns]

antes = len(df_clean)
df_clean = df_clean.dropna(subset=cols_clima_residual)
print(f"Filas eliminadas por nulos residuales: {antes - len(df_clean)} → Shape: {df_clean.shape}")

# Sincronizamos la eliminación de la fila en el DataFrame de metadatos utilizando el índice
metadata = metadata.loc[df_clean.index]
print("Shape de metadatos alineada:", metadata.shape)

Filas eliminadas por nulos residuales: 1 → Shape: (1172, 79)
Shape de metadatos alineada: (1172, 6)


### 7. Definición de Predictores, Target y Control de Fuga de Información
La variable objetivo del modelo de clasificación es `clase`. Para evitar cualquier fuga de información, excluimos las variables directamente asociadas a la construcción del target: `no3_mgl`, `afectada` y `en_riesgo`. 

Asimismo, excluimos de los predictores `X` las variables de metadatos que actúan como identificadores no descriptivos (`id_registro`, `ipa`, `fecha`), pero conservamos `lat`, `lon` y `anio` al considerarse variables espaciales y temporales relevantes en el modelado.

In [7]:
TARGET = 'clase'
cols_fuga = ['no3_mgl', 'afectada', 'en_riesgo']
cols_excluir_X = cols_fuga + [TARGET] + ['id_registro', 'ipa', 'fecha']
cols_excluir_X = [c for c in cols_excluir_X if c in df_clean.columns]

y = df_clean[TARGET]
X = df_clean.drop(columns=cols_excluir_X)

print(f"Target: {TARGET}")
print(f"Predictores (X): {X.shape[1]} columnas y {X.shape[0]} observaciones.")

Target: clase
Predictores (X): 73 columnas y 1172 observaciones.


### 8. División Estratificada Train/Test manteniendo Índices
Dividimos los predictores `X`, el target `y` y la tabla de metadatos `metadata` simultáneamente mediante una partición estratificada 80% entrenamiento y 20% prueba, asegurando que la proporción de clases se mantenga idéntica y sin reconstruir ni reiniciar los índices para conservar la trazabilidad de los registros.

In [8]:
X_train, X_test, y_train, y_test, metadata_train, metadata_test = train_test_split(
    X, y, metadata, test_size=0.2, random_state=42, stratify=y
)
print(f"Entrenamiento - X_train: {X_train.shape} | y_train: {y_train.shape} | metadata_train: {metadata_train.shape}")
print(f"Prueba        - X_test: {X_test.shape}   | y_test: {y_test.shape}   | metadata_test: {metadata_test.shape}")

Entrenamiento - X_train: (937, 73) | y_train: (937,) | metadata_train: (937, 6)
Prueba        - X_test: (235, 73)   | y_test: (235,)   | metadata_test: (235, 6)


### 9. Imputación de Valores Nulos Post-Split
Para garantizar la rigurosidad metodológica y evitar fuga de información desde el conjunto de prueba, todos los imputadores se ajustan (`fit`) exclusivamente con `X_train` y posteriormente se aplican (`transform`) a `X_train` y `X_test`.

*   **Variables CLC (`clc2012_nombre`, `clc2018_nombre`):** Imputación por la categoría más frecuente (moda).
*   **Variables de pH (`soil_phh2o_0_5cm`, `soil_phh2o_5_15cm`, `soil_phh2o_15_30cm`):** Imputación por la mediana.
*   **SIGPAC (`sigpac_coef_regadio`):** Se crea una columna indicadora binaria de la ausencia de dato original (`sigpac_coef_regadio_nulo`) en train y test, e imputamos los valores nulos en la columna original con 0.

In [9]:
# Imputación CLC (categóricas)
cols_clc = ['clc2012_nombre', 'clc2018_nombre']
imputer_clc = SimpleImputer(strategy='most_frequent')
X_train[cols_clc] = imputer_clc.fit_transform(X_train[cols_clc])
X_test[cols_clc] = imputer_clc.transform(X_test[cols_clc])

# Imputación pH (numéricas)
cols_ph = ['soil_phh2o_0_5cm', 'soil_phh2o_5_15cm', 'soil_phh2o_15_30cm']
cols_ph = [c for c in cols_ph if c in X_train.columns]
imputer_ph = SimpleImputer(strategy='median')
X_train[cols_ph] = imputer_ph.fit_transform(X_train[cols_ph])
X_test[cols_ph] = imputer_ph.transform(X_test[cols_ph])

# Imputación SIGPAC
if 'sigpac_coef_regadio' in X_train.columns:
    X_train['sigpac_coef_regadio_nulo'] = X_train['sigpac_coef_regadio'].isna().astype(int)
    X_test['sigpac_coef_regadio_nulo'] = X_test['sigpac_coef_regadio'].isna().astype(int)
    
    X_train['sigpac_coef_regadio'] = X_train['sigpac_coef_regadio'].fillna(0)
    X_test['sigpac_coef_regadio'] = X_test['sigpac_coef_regadio'].fillna(0)

print("Nulos remanentes en train:", X_train.isna().sum().sum())
print("Nulos remanentes en test:", X_test.isna().sum().sum())

Nulos remanentes en train: 0
Nulos remanentes en test: 0


### 10. Codificación de Variables Categóricas (One-Hot Encoding)
Transformamos las variables de texto en variables numéricas binarias (dummies) mediante `OneHotEncoder`, ajustándolo únicamente con el subconjunto de entrenamiento. La variable `municipio` se mantiene debido a que aporta información espacial-administrativa valiosa. Usamos `handle_unknown='ignore'` para que categorías nuevas en test no provoquen fallos.

In [10]:
cols_categoricas = X_train.select_dtypes(include=['object', 'string']).columns.tolist()
print("Variables categóricas a codificar:", cols_categoricas)
for col in cols_categoricas:
    print(f"  {col}: {X_train[col].nunique()} categorías")

# Ajuste del codificador en train
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train[cols_categoricas])

# Transformación y reconstrucción de DataFrames manteniendo índices
X_train_cat = pd.DataFrame(
    encoder.transform(X_train[cols_categoricas]),
    columns=encoder.get_feature_names_out(cols_categoricas),
    index=X_train.index
)
X_test_cat = pd.DataFrame(
    encoder.transform(X_test[cols_categoricas]),
    columns=encoder.get_feature_names_out(cols_categoricas),
    index=X_test.index
)

X_train = pd.concat([X_train.drop(columns=cols_categoricas), X_train_cat], axis=1)
X_test = pd.concat([X_test.drop(columns=cols_categoricas), X_test_cat], axis=1)

# Conversión de variables booleanas originales a enteros (0/1)
cols_bool = X_train.select_dtypes(include='bool').columns.tolist()
X_train[cols_bool] = X_train[cols_bool].astype(int)
X_test[cols_bool] = X_test[cols_bool].astype(int)

print(f"Train final tras codificar: {X_train.shape}")
print(f"Test final tras codificar: {X_test.shape}")

Variables categóricas a codificar: ['naturaleza', 'municipio', 'red', 'clc2012_nombre', 'clc2018_nombre', 'sigpac_uso_nombre']


  naturaleza: 3 categorías
  municipio: 55 categorías
  red: 2 categorías
  clc2012_nombre: 15 categorías
  clc2018_nombre: 15 categorías
  sigpac_uso_nombre: 14 categorías


Train final tras codificar: (937, 172)
Test final tras codificar: (235, 172)


### 11. Escalado de Variables Continuas
Escalamos mediante `StandardScaler` únicamente las variables continuas. Se excluyen del escalado todas las columnas binarias, booleanas y dummies del One-Hot para conservar su naturaleza interpretable de 0 y 1. 

El escalado se mantiene ya que el TFM incluye Regresión Logística (la cual es sensible a la escala de las variables), aunque otros modelos basados en árboles no lo requieran formalmente.

In [11]:
cols_numericas = X_train.select_dtypes(include=np.number).columns.tolist()

# Columnas binarias (dummies del One-Hot, booleanas originales e indicadores de nulo)
cols_binarias = [c for c in cols_numericas if X_train[c].nunique() <= 2]
cols_continuas = [c for c in cols_numericas if c not in cols_binarias]

print(f"Columnas binarias excluidas del escalado: {len(cols_binarias)}")
print(f"Columnas continuas a escalar: {len(cols_continuas)}")

# Ajuste y transformación en train, aplicación en test
scaler = StandardScaler()
X_train[cols_continuas] = scaler.fit_transform(X_train[cols_continuas])
X_test[cols_continuas] = scaler.transform(X_test[cols_continuas])

print("Escalado estándar aplicado correctamente.")

Columnas binarias excluidas del escalado: 113
Columnas continuas a escalar: 59
Escalado estándar aplicado correctamente.


### 12. Validación de Consistencia del Preprocesamiento
Realizamos verificaciones finales sobre las dimensiones, tipos de datos, ausencia de nulos y control de fugas de información, comprobando además la estratificación del target.

In [12]:
print("=== VERIFICACIÓN FINAL DEL PREPROCESAMIENTO ===")
print("Dimensiones X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Nulos X_train:", X_train.isna().sum().sum(), "| X_test:", X_test.isna().sum().sum())
print("Tipos no numéricos restantes en X_train:", X_train.select_dtypes(exclude=np.number).columns.tolist())
print("Columnas X_train == X_test:", list(X_train.columns) == list(X_test.columns))
print("Fuga de target detectada:", [c for c in ['no3_mgl', 'afectada', 'en_riesgo', 'clase'] if c in X_train.columns] or "Ninguna")

# Proporción de clases
dist_clase = pd.DataFrame({
    'train_%': y_train.value_counts(normalize=True)*100,
    'test_%': y_test.value_counts(normalize=True)*100
})
print("\nDistribución de clases (%):\n", dist_clase.round(2))

=== VERIFICACIÓN FINAL DEL PREPROCESAMIENTO ===
Dimensiones X_train: (937, 172) | X_test: (235, 172)
Nulos X_train: 0 | X_test: 0
Tipos no numéricos restantes en X_train: []
Columnas X_train == X_test: True
Fuga de target detectada: Ninguna

Distribución de clases (%):
           train_%  test_%
clase                    
afectada    50.37   50.64
normal      35.01   34.89
riesgo      14.62   14.47


### 13. Guardado de Datos y Transformadores del Preprocesamiento
Guardamos los conjuntos de datos, metadatos y los objetos transformadores entrenados en el directorio del proyecto.

In [13]:
carpeta_salida = BASE_DIR / '10_preprocesado_modelado'
(carpeta_salida / 'train').mkdir(parents=True, exist_ok=True)
(carpeta_salida / 'test').mkdir(parents=True, exist_ok=True)

# Guardar CSVs
X_train.to_csv(carpeta_salida / 'train' / 'X_train_larioja.csv', index=False)
y_train.to_csv(carpeta_salida / 'train' / 'y_train_larioja.csv', index=False)
X_test.to_csv(carpeta_salida / 'test' / 'X_test_larioja.csv', index=False)
y_test.to_csv(carpeta_salida / 'test' / 'y_test_larioja.csv', index=False)
metadata_train.to_csv(carpeta_salida / 'train' / 'metadata_train.csv', index=False)
metadata_test.to_csv(carpeta_salida / 'test' / 'metadata_test.csv', index=False)

# Guardar transformadores
joblib.dump(imputer_clc, carpeta_salida / 'imputer_categoricas.joblib')
joblib.dump(imputer_ph, carpeta_salida / 'imputer_ph.joblib')
joblib.dump(encoder, carpeta_salida / 'onehot_encoder.joblib')
joblib.dump(scaler, carpeta_salida / 'standard_scaler.joblib')

# Guardar archivos de configuración de columnas
columnas_finales = list(X_train.columns)
with open(carpeta_salida / 'columnas_finales.json', 'w') as f:
    json.dump(columnas_finales, f)

config_preprocesamiento = {
    "cols_categoricas": cols_categoricas,
    "cols_continuas": cols_continuas,
    "cols_binarias": cols_binarias,
    "target": TARGET
}
with open(carpeta_salida / 'config_preprocesamiento.json', 'w') as f:
    json.dump(config_preprocesamiento, f)

print(f"Se han exportado exitosamente todos los archivos y transformadores en: {carpeta_salida}")

Se han exportado exitosamente todos los archivos y transformadores en: C:\Users\mcangulo\OneDrive - FEDERACION DE EMPRESAS DE LA RIOJA\Escritorio\dataset_larioja\10_preprocesado_modelado


### Nota metodológica final del preprocesamiento
> El conjunto de entrenamiento conserva su distribución original. El desbalance de clases no se modifica durante el preprocesamiento. Su tratamiento se realiza exclusivamente en la etapa de modelado mediante comparación de modelos con y sin ponderación de clases.